In [4]:
#!/usr/bin/env python3
import os
import glob
import re
import numpy as np
import pandas as pd
import h5py
import matplotlib
import matplotlib.pyplot as plt
import logging

from bm3d import bm3d
from bm4d import bm4d

# ==========================================
# 0. CONFIGURATION 
# ==========================================
base_dir = "/pscratch/sd/k/kberard/SCGSR/Data/diamond_1x1x1_bfd/density_data/vmc_J2"
ref_path = os.path.join(base_dir, "density_tot_ref_mean.h5")
dft_path = '/global/u2/k/kberard/SCGSR/Research/Diamond/Data/density_tot_ref.h5'

# ==========================================
# 1. UNIFIED MATH & UTILITY FUNCTIONS
# ==========================================
def D_JS(p1, p2, tol=1e-16):
    """Calculates Jensen-Shannon Divergence."""
    p1 = p1 / (np.sum(p1) + 1e-16)
    p2 = p2 / (np.sum(p2) + 1e-16)
    pm = (p1 + p2) / 2
    p1_nonzero, p2_nonzero = np.abs(p1) > tol, np.abs(p2) > tol
    
    d = 0.5 * (
        (np.abs(p1[p1_nonzero]) * np.log(np.abs(p1[p1_nonzero]) / np.abs(pm[p1_nonzero]))).sum() + 
        (np.abs(p2[p2_nonzero]) * np.log(np.abs(p2[p2_nonzero]) / np.abs(pm[p2_nonzero]))).sum()
    )
    return d / np.log(2)

def transform(density, density_ref, transform_type='residual_noise'):
    if transform_type == 'residual_noise':
        return (density - density_ref) / (np.sqrt(np.abs(density_ref)) + 1e-12)
    return density

def inverse_transform(density_trans, density_ref, transform_type='residual_noise'):
    if transform_type == 'residual_noise':
        return density_ref + (np.sqrt(np.abs(density_ref)) * density_trans)
    return density_trans

def encode_voxel_to_rgb_global(vol_3d):
    """Global scaling prevents intensity bias, isolating spatial slicing bias."""
    v_min, v_max = float(vol_3d.min()), float(vol_3d.max())
    if v_max == v_min: v_max = v_min + 1e-6
    normed = (vol_3d - v_min) / (v_max - v_min)
    return np.stack([normed]*3, axis=-1).astype(np.float32), v_min, v_max

def decode_rgb_to_voxel_global(rgb_volume, v_min, v_max):
    return rgb_volume[:, :, :, 0] * (v_max - v_min) + v_min

def evaluate_and_enforce(denoised_d, ref_d):
    """Enforces non-negativity and exactly 8 electrons."""
    denoised_d = np.maximum(denoised_d, 0.0)
    denoised_d = denoised_d * (8.0 / (np.sum(denoised_d) + 1e-16))
    return D_JS(denoised_d, ref_d), denoised_d

# ==========================================
# 2. INFERENCE DISPATCHER (BM3D vs BM4D)
# ==========================================
def run_bm_inference(test_d, ref_d_dft, model_name, sample_count):
    # 1. Standardize into residual space
    trans_d = transform(test_d, ref_d_dft, 'residual_noise')
    
    # Calculate noise sigma specific to the sample count
    sigma_psd_bm3d = 0.1
    sigma_psd_bm4d = 0.001 * np.sqrt(6881280 / sample_count)

    if model_name == 'bm3d':
        # BM3D requires 2D slicing (XZ plane lost)
        rgb_vol, v_min, v_max = encode_voxel_to_rgb_global(trans_d)
        denoised_rgb = np.zeros_like(rgb_vol)
        
        for i in range(trans_d.shape[0]):
            denoised_gray = bm3d(rgb_vol[i, :, :, 0], sigma_psd=sigma_psd_bm3d)
            denoised_rgb[i] = np.stack([denoised_gray]*3, axis=-1)
            
        denoised_trans = decode_rgb_to_voxel_global(denoised_rgb, v_min, v_max)

    elif model_name == 'bm4d':
        # BM4D operates natively on the 3D volume
        denoised_trans = bm4d(trans_d, sigma_psd_bm4d)

    else:
        raise ValueError("Invalid model.")

    # 2. Inverse transform back to physical density
    return inverse_transform(denoised_trans, ref_d_dft, 'residual_noise')


# ==========================================
# 3. VISUALIZATION OF SLICING ARTIFACTS
# ==========================================
def plot_orthogonal_cross_section(bm3d_vol, bm4d_vol, ref_vol, sample_num):
    """
    Plots the XZ plane. Since BM3D sliced along the XY plane (Z-axis), 
    viewing it from the XZ perspective will expose the banding artifacts.
    """
    matplotlib.use('Agg')
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Extract the middle slice along the Y axis to view the XZ plane
    y_mid = 32
    eps = 1e-6
    
    # Use log scale for better contrast of the electron density
    d_ref  = np.log10(ref_vol[:, y_mid, :] + eps)
    d_bm3d = np.log10(bm3d_vol[:, y_mid, :] + eps)
    d_bm4d = np.log10(bm4d_vol[:, y_mid, :] + eps)
    
    vmin, vmax = np.min(d_ref), np.max(d_ref)
    
    axes[0].imshow(d_bm3d, cmap='inferno', origin='lower', vmin=vmin, vmax=vmax)
    axes[0].set_title("BM3D (2D Slicing)\nNotice banding along Z-axis", fontsize=14)
    axes[0].set_ylabel("Z-axis", fontsize=12)
    axes[0].set_xlabel("X-axis", fontsize=12)
    
    axes[1].imshow(d_bm4d, cmap='inferno', origin='lower', vmin=vmin, vmax=vmax)
    axes[1].set_title("BM4D (3D Volumetric)\nSmooth Isotropic Structure", fontsize=14)
    axes[1].set_xlabel("X-axis", fontsize=12)
    
    axes[2].imshow(d_ref, cmap='inferno', origin='lower', vmin=vmin, vmax=vmax)
    axes[2].set_title("VMC Reference", fontsize=14)
    axes[2].set_xlabel("X-axis", fontsize=12)
    
    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    plt.savefig(f"Slicing_Artifact_Comparison_Sample_{sample_num}.pdf", bbox_inches='tight')
    plt.close()

# ==========================================
# 4. MAIN EXECUTION PIPELINE
# ==========================================
def main():
    print("Loading Base Data & DFT Reference...")
    with h5py.File(ref_path, 'r') as file:
        ref_d_mean = file['density'][:]
        ref_d_mean = ref_d_mean * (8.0 / np.sum(ref_d_mean))

    with h5py.File(dft_path, 'r') as file:
        dft_d = file['density'][:]
        dft_d = dft_d * (8.0 / np.sum(dft_d))
        
    DFT_vs_VMC = D_JS(ref_d_mean, dft_d)
    noisy_files = sorted(glob.glob(os.path.join(base_dir, "density_tot_vmc_mean*.h5")))
    print(f"Found {len(noisy_files)} noisy files.")

    models_to_run = ['bm3d', 'bm4d']
    results_dict = {model: [] for model in models_to_run}
    results_ref = []

    print("\n=== Commencing Denoising Comparison ===")
    for noisy_path in noisy_files:
        match = re.search(r"(\d+)\.h5$", noisy_path)
        if not match: continue
        sample_num = int(match.group(1))
        
        with h5py.File(noisy_path, 'r') as file:
            test_d = file['density'][:]
            
        jsd_ref, _ = evaluate_and_enforce(test_d, ref_d_mean)
        results_ref.append((sample_num, jsd_ref))
        print(f"\n-> Sample {sample_num} loaded. Baseline JSD: {jsd_ref:.6e}")
        
        saved_vols = {}
        for model_name in models_to_run:
            raw_denoised = run_bm_inference(test_d, dft_d, model_name, sample_count=sample_num)
            jsd_score, clean_denoised = evaluate_and_enforce(raw_denoised, ref_d_mean)
            
            results_dict[model_name].append((sample_num, jsd_score))
            saved_vols[model_name] = clean_denoised
            print(f"   ↳ {model_name.upper()} JSD = {jsd_score:.6e}")
            
        # For the lowest sample count (noisiest), generate the artifact proof plot
        if sample_num == sorted([int(re.search(r"(\d+)\.h5$", f).group(1)) for f in noisy_files])[0]:
            print("   ↳ Generating spatial artifact proof visualization...")
            plot_orthogonal_cross_section(saved_vols['bm3d'], saved_vols['bm4d'], ref_d_mean, sample_num)

    # Generate Convergence Plot
    generate_log_plots(results_dict, results_ref, DFT_vs_VMC)

def generate_log_plots(results_dict, results_ref, DFT_vs_VMC):
    print("\n=== Generating Convergence Plot ===")
    df = pd.DataFrame(np.array(sorted(results_ref, key=lambda x: x[0])), columns=["Samples", "Noisy_VMC"])
    for model_name in ['bm3d', 'bm4d']:
        model_data = np.array(sorted(results_dict[model_name], key=lambda x: x[0]))
        temp_df = pd.DataFrame(model_data, columns=["Samples", model_name])
        df = pd.merge(df, temp_df, on="Samples", how="outer")

    df.to_csv("BM_Comparison_Performance.csv", index=False)

    matplotlib.use('Agg')
    plt.rcParams.update({
        'font.size': 14, 'font.family': 'serif', 'axes.labelsize': 16,
        'axes.linewidth': 1.5, 'xtick.major.size': 8, 'xtick.major.width': 1.5,
        'ytick.major.size': 8, 'ytick.major.width': 1.5,
        'legend.frameon': True, 'legend.edgecolor': 'black', 'legend.fontsize': 12
    })

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.plot(df["Samples"], df["Noisy_VMC"], marker="o", linestyle="--", color="gray", label="Noisy VMC Baseline", alpha=0.5, markersize=8)
    
    ax.plot(df["Samples"], df["bm3d"], marker="s", linestyle="--", color="blue", label="BM3D (2D Slicing)", linewidth=2.5, markersize=8)
    ax.plot(df["Samples"], df["bm4d"], marker="D", linestyle="-", color="green", label="BM4D (3D Volumetric)", linewidth=2.5, markersize=8)

    ax.axhline(DFT_vs_VMC, color="black", linestyle=":", label="DFT Baseline", linewidth=2)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Number of VMC Samples', fontweight='bold', labelpad=10)
    ax.set_ylabel(r'$D_{JS}$ (Jensen-Shannon Divergence)', fontweight='bold', labelpad=10)
    
    ax.tick_params(axis='both', which='minor', direction='in', length=4, width=1)
    ax.tick_params(axis='both', which='major', direction='in', length=8, width=1.5)
    
    ax.legend(loc='best')
    plt.tight_layout()
    plt.savefig("BM3D_vs_BM4D_Convergence.pdf", bbox_inches='tight')
    plt.close()

if __name__ == "__main__":
    main()

Loading Base Data & DFT Reference...
Found 33 noisy files.

=== Commencing Denoising Comparison ===

-> Sample 10240 loaded. Baseline JSD: 4.474564e-01
   ↳ BM3D JSD = 3.656943e-03
   ↳ BM4D JSD = 5.345214e-04
   ↳ Generating spatial artifact proof visualization...

-> Sample 20480 loaded. Baseline JSD: 2.926950e-01
   ↳ BM3D JSD = 1.761578e-03
   ↳ BM4D JSD = 2.280337e-04

-> Sample 40960 loaded. Baseline JSD: 1.693626e-01
   ↳ BM3D JSD = 9.353417e-04
   ↳ BM4D JSD = 1.111449e-04

-> Sample 81920 loaded. Baseline JSD: 9.013380e-02
   ↳ BM3D JSD = 4.377086e-04
   ↳ BM4D JSD = 8.357388e-05

-> Sample 163840 loaded. Baseline JSD: 4.466968e-02
   ↳ BM3D JSD = 2.578375e-04
   ↳ BM4D JSD = 7.711273e-05

-> Sample 327680 loaded. Baseline JSD: 2.115961e-02
   ↳ BM3D JSD = 1.355466e-04
   ↳ BM4D JSD = 6.335592e-05

-> Sample 430080 loaded. Baseline JSD: 1.575039e-02
   ↳ BM3D JSD = 1.240958e-04
   ↳ BM4D JSD = 5.320038e-05

-> Sample 655360 loaded. Baseline JSD: 9.959496e-03
   ↳ BM3D JSD = 9.